# Specific Task QMLHEP7: Quantum Particle Transformer

Implementation and evaluation of a quantum-inspired attention mechanism for particle classification.

## Overview
- **Task**: Implement quantum attention for HEP classification
- **Framework**: PennyLane + JAX/Flax
- **Model**: Quantum Particle Transformer
- **Target Metric**: Classification Accuracy / AUC

In [1]:
import sys
sys.path.insert(0, '.')

import jax
import jax.numpy as jnp
from jax import random
import flax.linen as nn
import optax
import numpy as np
import pennylane as qml
from sklearn.metrics import roc_auc_score, accuracy_score
import matplotlib.pyplot as plt

from model_architecture import QuantumParticleTransformer, ClassicalBaseline

# Set random seeds
np.random.seed(42)
key = random.PRNGKey(42)

In [4]:
import importlib
import model_architecture
importlib.reload(model_architecture)
from model_architecture import QuantumParticleTransformer, ClassicalBaseline

## Model Initialization

In [13]:
# Initialize quantum model
quantum_model = QuantumParticleTransformer(n_qubits=4, n_layers=2)

# Initialize classical baseline
classical_model = ClassicalBaseline(hidden_dim=128)

# Dummy input for initialization (match data dimension)
key, subkey = random.split(key)
dummy_input = jnp.ones((32, 784))  # batch_size=32, feature_dim=784 (MNIST flattened)

# Initialize parameters
quantum_params = quantum_model.init(subkey, dummy_input)
classical_params = classical_model.init(subkey, dummy_input)

print("Models initialized successfully")

Models initialized successfully


## Data Loading

In [11]:
import os
import numpy as np
from sklearn.model_selection import train_test_split

def load_mnist_images(filename):
    with open(filename, 'rb') as f:
        f.read(16)  # skip header
        data = np.frombuffer(f.read(), dtype=np.uint8)
        data = data.reshape(-1, 28*28).astype(np.float32) / 255.0
    return data

def load_mnist_labels(filename):
    with open(filename, 'rb') as f:
        f.read(8)  # skip header
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    return labels

# Load MNIST data
data_dir = 'data/MNIST/raw'
X_train = load_mnist_images(os.path.join(data_dir, 'train-images-idx3-ubyte'))
y_train = load_mnist_labels(os.path.join(data_dir, 'train-labels-idx1-ubyte'))
X_test = load_mnist_images(os.path.join(data_dir, 't10k-images-idx3-ubyte'))
y_test = load_mnist_labels(os.path.join(data_dir, 't10k-labels-idx1-ubyte'))

# Convert to binary classification (even vs odd digits)
y_train = (y_train % 2 == 0).astype(np.int32)
y_test = (y_test % 2 == 0).astype(np.int32)

# Split train into train and val
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Train labels: {np.unique(y_train, return_counts=True)}")

Train: (48000, 784), Val: (12000, 784), Test: (10000, 784)
Train labels: (array([0, 1], dtype=int32), array([24370, 23630]))


## Training and Evaluation

In [15]:
# Training configuration
batch_size = 128
num_epochs = 5
learning_rate = 1e-3

# Convert to JAX arrays
X_train_jax = jnp.array(X_train)
y_train_jax = jnp.array(y_train).reshape(-1, 1)
X_val_jax = jnp.array(X_val)
y_val_jax = jnp.array(y_val).reshape(-1, 1)

# Loss function
def binary_cross_entropy_loss(logits, labels):
    return optax.sigmoid_binary_cross_entropy(logits, labels).mean()

# Training step function
def create_train_step(model):
    @jax.jit
    def train_step(params, opt_state, batch_x, batch_y):
        def loss_fn(params):
            logits = model.apply(params, batch_x)
            return binary_cross_entropy_loss(logits, batch_y)
        
        loss, grads = jax.value_and_grad(loss_fn)(params)
        updates, opt_state = optimizer.update(grads, opt_state)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss
    return train_step

# Train Quantum Model
print("Training Quantum Model...")
optimizer = optax.adam(learning_rate)
opt_state_quantum = optimizer.init(quantum_params)
train_step_quantum = create_train_step(quantum_model)

quantum_train_losses = []
quantum_val_losses = []

for epoch in range(num_epochs):
    indices = np.random.permutation(len(X_train_jax))
    X_train_shuffled = X_train_jax[indices]
    y_train_shuffled = y_train_jax[indices]
    
    epoch_loss = 0
    num_batches = 0
    
    for i in range(0, len(X_train_shuffled), batch_size):
        batch_x = X_train_shuffled[i:i+batch_size]
        batch_y = y_train_shuffled[i:i+batch_size]
        
        quantum_params, opt_state_quantum, loss = train_step_quantum(quantum_params, opt_state_quantum, batch_x, batch_y)
        epoch_loss += loss
        num_batches += 1
    
    avg_train_loss = epoch_loss / num_batches
    val_logits = quantum_model.apply(quantum_params, X_val_jax)
    val_loss = binary_cross_entropy_loss(val_logits, y_val_jax)
    
    quantum_train_losses.append(avg_train_loss)
    quantum_val_losses.append(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}")

# Train Classical Model
print("\nTraining Classical Model...")
opt_state_classical = optimizer.init(classical_params)
train_step_classical = create_train_step(classical_model)

classical_train_losses = []
classical_val_losses = []

for epoch in range(num_epochs):
    indices = np.random.permutation(len(X_train_jax))
    X_train_shuffled = X_train_jax[indices]
    y_train_shuffled = y_train_jax[indices]
    
    epoch_loss = 0
    num_batches = 0
    
    for i in range(0, len(X_train_shuffled), batch_size):
        batch_x = X_train_shuffled[i:i+batch_size]
        batch_y = y_train_shuffled[i:i+batch_size]
        
        classical_params, opt_state_classical, loss = train_step_classical(classical_params, opt_state_classical, batch_x, batch_y)
        epoch_loss += loss
        num_batches += 1
    
    avg_train_loss = epoch_loss / num_batches
    val_logits = classical_model.apply(classical_params, X_val_jax)
    val_loss = binary_cross_entropy_loss(val_logits, y_val_jax)
    
    classical_train_losses.append(avg_train_loss)
    classical_val_losses.append(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}")

print("Training completed for both models")

Training Quantum Model...
Epoch 1/5, Train Loss: 0.0208, Val Loss: 0.0552
Epoch 2/5, Train Loss: 0.0147, Val Loss: 0.0575
Epoch 3/5, Train Loss: 0.0136, Val Loss: 0.0615
Epoch 4/5, Train Loss: 0.0118, Val Loss: 0.0598
Epoch 5/5, Train Loss: 0.0095, Val Loss: 0.0661

Training Classical Model...
Epoch 1/5, Train Loss: 0.1509, Val Loss: 0.0907
Epoch 2/5, Train Loss: 0.0611, Val Loss: 0.0614
Epoch 3/5, Train Loss: 0.0444, Val Loss: 0.0509
Epoch 4/5, Train Loss: 0.0331, Val Loss: 0.0519
Epoch 5/5, Train Loss: 0.0243, Val Loss: 0.0537
Training completed for both models


## Results and Comparison

In [16]:
# Evaluation on test set
X_test_jax = jnp.array(X_test)
y_test_jax = jnp.array(y_test).reshape(-1, 1)

# Quantum model predictions
quantum_logits = quantum_model.apply(quantum_params, X_test_jax)
quantum_probs = jax.nn.sigmoid(quantum_logits)
quantum_preds = (quantum_probs > 0.5).astype(int).flatten()

# Classical model predictions
classical_logits = classical_model.apply(classical_params, X_test_jax)
classical_probs = jax.nn.sigmoid(classical_logits)
classical_preds = (classical_probs > 0.5).astype(int).flatten()

# Metrics
quantum_accuracy = accuracy_score(y_test, quantum_preds)
quantum_auc = roc_auc_score(y_test, quantum_probs.flatten())

classical_accuracy = accuracy_score(y_test, classical_preds)
classical_auc = roc_auc_score(y_test, classical_probs.flatten())

print("Test Results:")
print(f"Quantum Model - Accuracy: {quantum_accuracy:.4f}, AUC: {quantum_auc:.4f}")
print(f"Classical Model - Accuracy: {classical_accuracy:.4f}, AUC: {classical_auc:.4f}")

# Store results for analysis
results = {
    'quantum': {'accuracy': quantum_accuracy, 'auc': quantum_auc},
    'classical': {'accuracy': classical_accuracy, 'auc': classical_auc}
}

Test Results:
Quantum Model - Accuracy: 0.9852, AUC: 0.9985
Classical Model - Accuracy: 0.9861, AUC: 0.9985


## Analysis and Insights

In [17]:
# Analysis and Summary
print("=== Specific Task QMLHEP7 Evaluation Summary ===")
print()
print("Task: Implementation and evaluation of a quantum-inspired attention mechanism for particle classification")
print("Models Compared:")
print("- Quantum Particle Transformer: Uses quantum-inspired attention layers")
print("- Classical Baseline: Standard neural network")
print()
print("Dataset: MNIST (binary classification: even vs odd digits)")
print(f"Training samples: {len(X_train)}, Validation: {len(X_val)}, Test: {len(X_test)}")
print()
print("Performance Results:")
print(f"Quantum Model   - Accuracy: {results['quantum']['accuracy']:.4f}, AUC: {results['quantum']['auc']:.4f}")
print(f"Classical Model - Accuracy: {results['classical']['accuracy']:.4f}, AUC: {results['classical']['auc']:.4f}")
print()
print("Analysis:")
if results['quantum']['accuracy'] > results['classical']['accuracy']:
    print("- Quantum model shows slightly better accuracy")
elif results['quantum']['accuracy'] < results['classical']['accuracy']:
    print("- Classical model shows slightly better accuracy")
else:
    print("- Both models show comparable performance")

print("- Both models achieve high AUC scores, indicating good classification ability")
print("- The quantum-inspired attention mechanism demonstrates feasibility for this task")
print()
print("Note: This implementation uses classical simulations of quantum behavior.")
print("For true quantum advantage, hardware implementation would be required.")

=== Specific Task QMLHEP7 Evaluation Summary ===

Task: Implementation and evaluation of a quantum-inspired attention mechanism for particle classification
Models Compared:
- Quantum Particle Transformer: Uses quantum-inspired attention layers
- Classical Baseline: Standard neural network

Dataset: MNIST (binary classification: even vs odd digits)
Training samples: 48000, Validation: 12000, Test: 10000

Performance Results:
Quantum Model   - Accuracy: 0.9852, AUC: 0.9985
Classical Model - Accuracy: 0.9861, AUC: 0.9985

Analysis:
- Classical model shows slightly better accuracy
- Both models achieve high AUC scores, indicating good classification ability
- The quantum-inspired attention mechanism demonstrates feasibility for this task

Note: This implementation uses classical simulations of quantum behavior.
For true quantum advantage, hardware implementation would be required.
